In [2]:
%load_ext autoreload
%autoreload 2
import os
print(os.getcwd())
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
import pickle
from torch import nn, optim
from torch.optim import Adam,AdamW

import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt

import anndata
print(anndata.__version__)
import anndata._core.file_backing
import weakref

from utils import simdatset, reproducibility
reproducibility(2025)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/home/hx/J/DADA/assay_upload
0.10.8


# Data load

In [3]:
out_pth = f"../result/"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 256

def _patched_setstate(self, state):
    self.__dict__ = state.copy()
    # Check if the required key exists
    if "_adata_ref" in state:
        self.__dict__["_adata_ref"] = weakref.ref(state["_adata_ref"])
    else:
        # If missing (old version), set a dummy value to bypass the crash
        # NOTE: This allows loading, but 'backed' features might be unstable.
        self.__dict__["_adata_ref"] = lambda: None 

filepath = os.path.join("../result/", "Stim_data_sim.pkl")
    
print("Patch applied. You can now run pickle.load()")
with open(filepath, 'rb') as file:
    loaded_data = pickle.load(file)
simudata_GTE = loaded_data['simudata_GTE']
simudata_HPA = loaded_data['simudata_HPA']
label_GTE = loaded_data['label_GTE']
label_HPA = loaded_data['label_HPA']
GTE_x_train = loaded_data['GTE_x_train']
GTE_x_val = loaded_data['GTE_x_val']
GTE_x_test = loaded_data['GTE_x_test']
GTE_y_train = loaded_data['GTE_y_train']
GTE_y_val = loaded_data['GTE_y_val']
GTE_y_test = loaded_data['GTE_y_test']
HPA_x = loaded_data['HPA_x']
HPA_y = loaded_data['HPA_y']
real_x = loaded_data['real_x']
all_genename = loaded_data['all_genename']
celltypes = loaded_data['celltypes']

model_path_base = out_pth + "model_stage1"
genename_df = pd.DataFrame(all_genename, columns=["GeneName"])
celltypes_df = pd.DataFrame(celltypes, columns=["TissueType"])

Patch applied. You can now run pickle.load()


# Models

## Stage1 Model

In [ ]:
%load_ext autoreload
%autoreload 2
from train_re import alternate_training_earlyStop, AdaptiveTAPEandDiffusion3

model = AdaptiveTAPEandDiffusion3(GTE_x_train.shape[1], GTE_y_train.shape[1],2, T=2000).to(device)

optimizer_main = AdamW([
    {'params': model.encoder.parameters()},
    {'params': model.predictor.parameters()},
    {'params': model.decoder.parameters()}], lr=1e-4)
optimizer_diffusion =AdamW(model.ref_creator.parameters(), lr=1e-3)
optimizer_all = AdamW(model.parameters(),1e-4)
epochs_main = 400 
epochs_diffusion = 1500

train_loader = DataLoader(simdatset(GTE_x_train, GTE_y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(simdatset(GTE_x_val, GTE_y_val), batch_size=batch_size, shuffle=False)

model, main_loss, diffloss = alternate_training_earlyStop(model, train_loader, val_loader, optimizer_main, optimizer_diffusion, 
                                                 epochs_main, epochs_diffusion, device=device,
                                                 patience=5, early_stop_start=400, early_stop_interval=50)
torch.save(model, model_path_base + ".pth")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Epoch [1/400], Main Loss: 0.0466
Epoch [2/400], Main Loss: 0.0354
Epoch [3/400], Main Loss: 0.0320
Epoch [4/400], Main Loss: 0.0305
Epoch [5/400], Main Loss: 0.0295
Epoch [6/400], Main Loss: 0.0287
Epoch [7/400], Main Loss: 0.0281
Epoch [8/400], Main Loss: 0.0276
Epoch [9/400], Main Loss: 0.0271
Epoch [10/400], Main Loss: 0.0267
Epoch [11/400], Main Loss: 0.0264
Epoch [12/400], Main Loss: 0.0260
Epoch [13/400], Main Loss: 0.0258
Epoch [14/400], Main Loss: 0.0255
Epoch [15/400], Main Loss: 0.0253
Epoch [16/400], Main Loss: 0.0251
Epoch [17/400], Main Loss: 0.0250
Epoch [18/400], Main Loss: 0.0247
Epoch [19/400], Main Loss: 0.0246
Epoch [20/400], Main Loss: 0.0244
Epoch [21/400], Main Loss: 0.0243
Epoch [22/400], Main Loss: 0.0242
Epoch [23/400], Main Loss: 0.0240
Epoch [24/400], Main Loss: 0.0239
Epoch [25/400], Main Loss: 0.0238
Epoch [26/400], Main Loss: 0.0238
Epoch [27/400], Main Loss: 0.0237
Epo

## GTE test, by Sigmatrix

In [ ]:
%load_ext autoreload
%autoreload 2
from train_re import evaluation, get_frac_from_sigmatrix
from utils import realdatset, calculate_evaluation_metrics

model = torch.load(model_path_base + ".pth", weights_only=False, map_location=device)
train_loader2 = DataLoader(simdatset(GTE_x_train, GTE_y_train), batch_size=batch_size, shuffle=False)
x_recon_tr, f_tr, z_tr  = evaluation(train_loader2, model, device=device)
sigmatrix = np.linalg.pinv(f_tr) @ x_recon_tr 
pd.DataFrame(sigmatrix).to_csv(out_pth + 'sigmatrix.csv', index=False, header=True)

test_loader =DataLoader(realdatset(GTE_x_test), batch_size=GTE_x_test.shape[0], shuffle=False)
model.eval()
model.state = 'test'

for _, X in enumerate(test_loader):
    X = X.to(device)
    z = model.encode(X).detach().to(device)
    pred_f = get_frac_from_sigmatrix(model, z, sigmatrix).numpy()
    # x_recon = model.decode(z).detach().cpu()
    x_recon = pred_f @ sigmatrix

## GTE test, AE

In [ ]:
%load_ext autoreload
%autoreload 2
from train_re import evaluation
from utils import calculate_evaluation_metrics

model = torch.load(model_path_base + ".pth", weights_only=False, map_location=device)
test_loader = DataLoader(simdatset(GTE_x_test, GTE_y_test), batch_size=batch_size, shuffle=False)
x_recon_te, f_te, z_te  = evaluation(test_loader, model, device=device)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## HPA，AE

In [ ]:
%load_ext autoreload
%autoreload 2
from train_re import adaptive_stage_domain_difSAwiwoS_noise
x_recon_HPA, f_HPA, z_HPA, model2 = adaptive_stage_domain_difSAwiwoS_noise(x=HPA_x, model_name= model_path_base,  adaptive=False, device=device)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Detected Latent Dimension: 256 (Original Gene Dim: 18757)


## HPA，by Sigmatrix

In [ ]:
from utils import realdatset
from train_re import get_frac_from_sigmatrix
model.eval()
model.state = 'test'
test_loader =DataLoader(realdatset(HPA_x), batch_size=HPA_x.shape[0], shuffle=False)
for _, X in enumerate(test_loader):
    X = X.to(device)
    z = model.encode(X).detach().to(device)
    pred_f = get_frac_from_sigmatrix(model, z, sigmatrix).numpy()

## HPA, original GTE data

In [ ]:
%load_ext autoreload
%autoreload 2
from train_re import adaptive_stage_domain_difSAwiwoS_noise

print(">>> Test result on Origin GTE ...")
x_recon_ori, f_ori, z_ori, model_stage2 = adaptive_stage_domain_difSAwiwoS_noise(
    x=HPA_x, 
    sour_x_train = GTE_x_train,
    model_name=model_path_base, 
    adaptive=True, 
    mode='overall5', 
    steps=10, 
    max_iter=40, 
    device=device, 
    sigmatrix=sigmatrix,
    generator=None,
    modeS="wo")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
>>> Test result on Origin GTE ...
Detected Latent Dimension: 256 (Original Gene Dim: 18757)
4000
50
Iter [10/40]
Iter [20/40]
Iter [30/40]
Iter [40/40]


## HPA, DADA

In [18]:
%load_ext autoreload
%autoreload 2
from train_re import adaptive_stage_domain_difSAwiwoS_latentMMD
model_path_base2 = out_pth + "model_stage2"

x_recon_diff, f_diff, z_diff, model_stage2 = adaptive_stage_domain_difSAwiwoS_latentMMD(
    x=HPA_x, 
    model_name=model_path_base, 
    adaptive=True, 
    steps=10, 
    max_iter=40, 
    device=device, 
    sigmatrix=None,
    generator=None,
    modeS="wo",
    save_name = model_path_base2 + "_DA")

[autoreload of train_re failed: Traceback (most recent call last):
  File "/home/hx/anaconda3/envs/dada/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/home/hx/anaconda3/envs/dada/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/home/hx/anaconda3/envs/dada/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/home/hx/anaconda3/envs/dada/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 349, in update_class
    if update_generic(old_obj, new_obj):
  File "/home/hx/anaconda3/envs/dada/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/home/hx/anaconda3/envs/dada/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 349, in update_class
    if update_generic(ol

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Detected Latent Dimension: 256
>>> Pre-generating Source Latents via Diffusion...
Iter [10/40]
Iter [20/40]
Iter [30/40]
Iter [40/40]
